# Skyntri — Model B: Skin Disease Screening Support (EfficientNet, face + body)

Model B is a **separate model** from Model A (facial concerns). It assesses images for a small, explicitly supported set of disease patterns on the **face or body**, and it flags lesions that need a doctor's assessment.

> **Safety rules built into this notebook and the backend**
> * Model B never diagnoses skin cancer and never rules it out. The *Suspicious lesion* class is a **referral flag**, tuned for high sensitivity.
> * A negative or "Other" result is **not** medical clearance.
> * The backend keeps Model B switched off until `DISEASE_SCREENING_ENABLED=true`. Set that only after dermatologist review of the evaluation in section 10.

**This notebook has not been run.** Upload the data, then run it on Colab with a GPU.

| Step | Section |
|---|---|
| 0 | Setup and configuration (taxonomy) |
| 1 | Download/load SCIN, Fitzpatrick17k, PAD-UFES-20 (+ DDI for external test only) |
| 2 | Cleaning: unreadable, duplicates/near-duplicates, conflicting labels, quality |
| 3 | Resize and cache, class balancing |
| 4 | Statistics (class × skin type × source) |
| 5 | Patient/case-grouped, stratified split |
| 6 | `tf.data` pipeline and augmentation |
| 7 | EfficientNet softmax model |
| 8 | Training (run yourself) |
| 9 | Operating points: referral threshold and abstention threshold (validation only) |
| 10 | Test evaluation: sensitivity/specificity/PPV/NPV, balanced accuracy, AUC, calibration, bootstrap CIs, Fitzpatrick subgroups, false-negative review, external DDI test |
| 11 | Export for the backend |
| 12 | Safe prediction function (same as the backend) |

## Datasets (download links and licences)

| Role | Dataset | Link | Licence / access | Why |
|---|---|---|---|---|
| **Primary** | **SCIN** (Google + Stanford), 10,000+ consented images, dermatologist labels, Fitzpatrick + Monk skin tone, face and body | https://github.com/google-research-datasets/scin (data: `gs://dx-scin-public-data/dataset`) | SCIN Data Use License (read it; no re-identification) | Real phone photos of common conditions (eczema, dermatitis, fungal, urticaria, psoriasis), diverse skin tones |
| Supplement | **Fitzpatrick17k**, 16,577 images, 114 conditions, Fitzpatrick labels | https://github.com/mattgroh/fitzpatrick17k (CSV; images via the form in the README) | CC BY-NC-SA 3.0 (non-commercial) | More examples per disease, skin-type labels for fairness checks |
| Referral pathway | **PAD-UFES-20**, 2,298 smartphone lesion images (BCC, SCC, MEL, ACK, NEV, SEK), biopsy info | https://data.mendeley.com/datasets/zr7vgbcyr2/1 | CC BY 4.0 | Smartphone images of skin cancers and precancers for the *Suspicious lesion* flag |
| **External test only** | **DDI** (Stanford), 656 biopsy-confirmed images, Fitzpatrick I–VI balanced | https://aimi.stanford.edu/datasets/ddi-diverse-dermatology-images | Research use agreement, non-commercial, no derivatives | Independent fairness test; **never train on it** |

Optional: ISIC Archive (https://www.isic-archive.com) for more lesion images. These are mostly dermoscopy images, which differ from phone photos, so use them with care.

## 0. Setup

In [ ]:
try:
    from google.colab import auth, drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

!pip -q install "tensorflow==2.20.0" "opencv-python-headless<5" scikit-learn pandas matplotlib seaborn tqdm

In [ ]:
import ast
import hashlib
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from PIL import Image, ImageOps
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
    top_k_accuracy_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ============================== CONFIGURATION ==============================
SEED = 42
DATA_ROOT = Path("/content/drive/MyDrive/skyntri/datasets/disease")
WORK_DIR = Path("/content/drive/MyDrive/skyntri/work/disease")
EXPORT_DIR = Path("/content/drive/MyDrive/skyntri/export")

SCIN_ROOT = DATA_ROOT / "scin"            # contains dataset/scin_cases.csv, dataset/scin_labels.csv, dataset/images/
FITZ_CSV = DATA_ROOT / "fitzpatrick17k/fitzpatrick17k.csv"
FITZ_IMAGES = DATA_ROOT / "fitzpatrick17k/images"      # <md5hash>.jpg
PAD_ROOT = DATA_ROOT / "pad_ufes_20"     # metadata.csv + images (any sub-folders)
DDI_ROOT = DATA_ROOT / "ddi"             # ddi_metadata.csv + images  (external TEST ONLY)

BACKBONE = "EfficientNetB0"
IMG_SIZE = 224 if BACKBONE == "EfficientNetB0" else 300
PRETRAINED_WEIGHTS = "imagenet"
BATCH_SIZE = 32
HEAD_EPOCHS = 8
FINE_TUNE_EPOCHS = 30
HEAD_LR = 1e-3
FINE_TUNE_LR = 3e-5
UNFREEZE_LAST_N_LAYERS = 100
LABEL_SMOOTHING = 0.05

MODEL_VERSION = f"skin-disease-{BACKBONE.lower()}-v1"
DATASET_VERSION = time.strftime("%Y-%m-%d")

SCIN_MIN_WEIGHT = 0.5          # keep a SCIN case only if its top dermatologist label has weight >= this
MIN_SIDE = 96
NEAR_DUP_HAMMING = 4
CACHE_SIDE = 448
OTHER_MAX_FRACTION = 0.35      # cap "Other" so it does not dominate training
TARGET_REFERRAL_SENSITIVITY = 0.95   # operating point for the Suspicious lesion flag (on validation)
TARGET_SELECTIVE_ACCURACY = 0.80     # abstain when the top probability is below the level reaching this

random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
for p in (WORK_DIR, EXPORT_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ---------------- Taxonomy (start small; add a class only when data + review support it) ----------------
CLASSES = [
    "Eczema / dermatitis",
    "Psoriasis",
    "Fungal infection",
    "Urticaria (hives)",
    "Suspicious lesion",           # referral class: melanoma, BCC, SCC, actinic keratosis
    "Other / not supported",
]
REFERRAL_CLASSES = ["Suspicious lesion"]
OTHER = "Other / not supported"
NUM_CLASSES = len(CLASSES)
CLASS_INDEX = {c: i for i, c in enumerate(CLASSES)}

# keyword -> class; checked in this order (first match wins)
TAXONOMY_RULES = [
    ("Suspicious lesion", ["melanoma", "basal cell", "squamous cell", "carcinoma", "actinic keratosis",
                           "bowen", "keratoacanthoma", "lentigo maligna"]),
    ("Psoriasis", ["psoriasis"]),
    ("Fungal infection", ["tinea", "fungal", "candid", "pityriasis versicolor", "ringworm", "onychomycosis"]),
    ("Urticaria (hives)", ["urticaria", "hives"]),
    ("Eczema / dermatitis", ["eczema", "dermatitis", "atopic", "lichen simplex", "dyshidro", "nummular"]),
]
PAD_CODES = {"MEL": "Suspicious lesion", "BCC": "Suspicious lesion", "SCC": "Suspicious lesion",
             "ACK": "Suspicious lesion", "NEV": OTHER, "SEK": OTHER}


def map_condition(name):
    text = str(name).lower()
    for cls, keys in TAXONOMY_RULES:
        if any(k in text for k in keys):
            return cls
    return OTHER


def fst_group(value):
    """Map Fitzpatrick 1-6 (various spellings) to I-II / III-IV / V-VI / unknown."""
    try:
        v = int(str(value).upper().replace("FST", "").strip())
    except ValueError:
        return "unknown"
    return {1: "I-II", 2: "I-II", 3: "III-IV", 4: "III-IV", 5: "V-VI", 6: "V-VI"}.get(v, "unknown")

## 1. Download and load

In [ ]:
# SCIN download (Colab). The bucket is public but gsutil needs a signed-in Google account.
# Run once, then copy to Drive if you want to keep it.
if IN_COLAB and not (SCIN_ROOT / "dataset/scin_cases.csv").exists():
    auth.authenticate_user()
    SCIN_ROOT.mkdir(parents=True, exist_ok=True)
    !gsutil -m -q cp -r gs://dx-scin-public-data/dataset "{SCIN_ROOT}/"

In [ ]:
def load_scin():
    cases_csv, labels_csv = SCIN_ROOT / "dataset/scin_cases.csv", SCIN_ROOT / "dataset/scin_labels.csv"
    if not cases_csv.exists():
        print("SCIN not found - skipped"); return []
    cases = pd.read_csv(cases_csv, dtype={"case_id": str})
    labels = pd.read_csv(labels_csv, dtype={"case_id": str})
    df = cases.merge(labels, on="case_id", how="inner")
    fst_cols = [c for c in df.columns if "fitzpatrick" in c.lower()]
    # prefer dermatologist-estimated skin type, fall back to self-reported
    fst_cols.sort(key=lambda c: (0 if "dermatologist" in c.lower() else 1, c))
    rows = []
    for _, r in df.iterrows():
        raw = r.get("weighted_skin_condition_label")
        if not isinstance(raw, str) or not raw.strip():
            continue
        try:
            weights = ast.literal_eval(raw)
        except (ValueError, SyntaxError):
            continue
        if not weights:
            continue
        top, w = max(weights.items(), key=lambda kv: kv[1])
        if w < SCIN_MIN_WEIGHT:
            continue
        fst = next((r[c] for c in fst_cols if pd.notna(r[c]) and str(r[c]) not in ("NONE_SELECTED", "")), None)
        for col in ("image_1_path", "image_2_path", "image_3_path"):
            p = r.get(col)
            if isinstance(p, str) and p:
                rows.append({"path": str(SCIN_ROOT / p), "source": "scin", "group": f"scin:{r['case_id']}",
                             "condition": top, "label": map_condition(top), "fst": fst_group(fst)})
    return rows


def load_fitz17k():
    if not FITZ_CSV.exists():
        print("Fitzpatrick17k not found - skipped"); return []
    df = pd.read_csv(FITZ_CSV)
    fst_col = "fitzpatrick_scale" if "fitzpatrick_scale" in df.columns else "fitzpatrick"
    rows = []
    for _, r in df.iterrows():
        path = FITZ_IMAGES / f"{r['md5hash']}.jpg"
        rows.append({"path": str(path), "source": "fitzpatrick17k", "group": f"fitz:{r['md5hash']}",
                     "condition": r["label"], "label": map_condition(r["label"]), "fst": fst_group(r[fst_col])})
    return rows


def load_pad_ufes():
    meta = PAD_ROOT / "metadata.csv"
    if not meta.exists():
        print("PAD-UFES-20 not found - skipped"); return []
    df = pd.read_csv(meta)
    index = {p.name: p for p in PAD_ROOT.rglob("*.png")}
    rows = []
    for _, r in df.iterrows():
        p = index.get(str(r["img_id"]))
        if p is None:
            continue
        rows.append({"path": str(p), "source": "pad_ufes_20", "group": f"pad:{r['patient_id']}",
                     "condition": r["diagnostic"], "label": PAD_CODES.get(r["diagnostic"], OTHER),
                     "fst": fst_group(r.get("fitspatrick"))})
    return rows


rows = load_scin() + load_fitz17k() + load_pad_ufes()
manifest = pd.DataFrame(rows)
print(manifest.groupby(["source", "label"]).size().unstack(fill_value=0))

## 2. Cleaning

In [ ]:
def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def dhash_hex(rgb, size=8):
    g = cv2.resize(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), (size + 1, size), interpolation=cv2.INTER_AREA)
    return np.packbits((g[:, 1:] > g[:, :-1]).flatten()).tobytes().hex()


def inspect(path):
    try:
        with Image.open(path) as im:
            rgb = np.asarray(ImageOps.exif_transpose(im).convert("RGB"))
    except Exception as exc:
        return {"ok": False, "reason": f"unreadable: {type(exc).__name__}"}
    h, w = rgb.shape[:2]
    if min(h, w) < MIN_SIDE:
        return {"ok": False, "reason": "too_small"}
    g = cv2.resize(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), (256, 256))
    return {"ok": True, "reason": None, "md5": file_md5(path), "dhash": dhash_hex(rgb),
            "luma": float(g.mean()), "sharpness": float(cv2.Laplacian(g, cv2.CV_64F).var())}


info = pd.DataFrame([inspect(p) for p in tqdm(manifest["path"], desc="inspecting")])
manifest = pd.concat([manifest.reset_index(drop=True), info], axis=1)
print(manifest["reason"].value_counts())
manifest = manifest[manifest["ok"] == True].copy()  # noqa: E712

bad = (manifest["luma"] < 25) | (manifest["luma"] > 245) | (manifest["sharpness"] < 5)
print("Removed for quality:", int(bad.sum()))
manifest = manifest[~bad]

# Exact duplicates: keep one; drop if the copies disagree on the label (ambiguous)
conflicting = manifest.groupby("md5")["label"].nunique()
conflicting = set(conflicting[conflicting > 1].index)
print("Exact duplicates with conflicting labels dropped:", len(conflicting))
manifest = manifest[~manifest["md5"].isin(conflicting)].drop_duplicates("md5").reset_index(drop=True)

In [ ]:
# Near duplicates -> same group (so they never cross train/val/test)
parent = {g: g for g in manifest["group"]}


def find(g):
    while parent[g] != g:
        parent[g] = parent[parent[g]]; g = parent[g]
    return g


hashes = np.array([np.frombuffer(bytes.fromhex(h), dtype=np.uint8) for h in manifest["dhash"]])
bits = np.unpackbits(hashes, axis=1)
groups = manifest["group"].tolist()
for band in range(4):
    buckets = defaultdict(list)
    for i, row in enumerate(hashes):
        buckets[bytes(row[band * 2: band * 2 + 2])].append(i)
    for idx in buckets.values():
        sub = bits[idx]
        for a in range(len(idx) - 1):
            dist = (sub[a + 1:] != sub[a]).sum(1)
            for off in np.nonzero(dist <= NEAR_DUP_HAMMING)[0]:
                ra, rb = find(groups[idx[a]]), find(groups[idx[a + 1 + off]])
                if ra != rb:
                    parent[rb] = ra
manifest["group"] = [find(g) for g in groups]

# A group must have one label; drop mixed groups
mixed = manifest.groupby("group")["label"].nunique()
mixed = set(mixed[mixed > 1].index)
print("Near-duplicate groups with mixed labels dropped:", len(mixed))
manifest = manifest[~manifest["group"].isin(mixed)].reset_index(drop=True)
print("Images:", len(manifest), "| groups:", manifest["group"].nunique())

## 3. Cache resized images and balance the *Other* class

In [ ]:
CACHE_DIR = WORK_DIR / "cleaned"; CACHE_DIR.mkdir(parents=True, exist_ok=True)


def cache_image(row):
    out = CACHE_DIR / f"{row['md5']}.jpg"
    if not out.exists():
        with Image.open(row["path"]) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            im.thumbnail((CACHE_SIDE, CACHE_SIDE), Image.Resampling.LANCZOS)
            im.save(out, quality=95)
    return str(out)


manifest["clean_path"] = [cache_image(r) for r in tqdm(manifest.to_dict("records"), desc="caching")]

# Cap "Other" (group-wise, so whole groups are kept or removed)
n_other_max = int(OTHER_MAX_FRACTION * len(manifest))
other_groups = manifest.loc[manifest.label == OTHER, "group"].drop_duplicates().sample(frac=1, random_state=SEED)
keep, count = set(), 0
sizes = manifest.groupby("group").size()
for g in other_groups:
    if count + sizes[g] > n_other_max:
        continue
    keep.add(g); count += sizes[g]
manifest = manifest[(manifest.label != OTHER) | manifest.group.isin(keep)].reset_index(drop=True)
manifest["y"] = manifest["label"].map(CLASS_INDEX)
manifest.to_pickle(WORK_DIR / "manifest_clean.pkl")

## 4. Statistics

In [ ]:
display(manifest.groupby(["label", "fst"]).size().unstack(fill_value=0))
display(manifest.groupby(["label", "source"]).size().unstack(fill_value=0))
manifest["label"].value_counts().plot.barh(title="Images per class"); plt.show()

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for ax, (_, r) in zip(axes.flat, manifest.groupby("label").head(3).iterrows()):
    ax.imshow(Image.open(r["clean_path"])); ax.axis("off"); ax.set_title(f"{r['label']}\n{r['source']}", fontsize=8)
plt.tight_layout(); plt.show()

## 5. Grouped, stratified split (cases/patients never cross splits)

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
folds = list(sgkf.split(manifest, manifest["y"], manifest["group"]))
manifest["split"] = "train"
manifest.loc[folds[0][1], "split"] = "test"
manifest.loc[folds[1][1], "split"] = "val"
assert manifest.groupby("group")["split"].nunique().max() == 1
display(manifest.groupby(["label", "split"]).size().unstack(fill_value=0))
manifest.to_csv(WORK_DIR / "manifest_split.csv", index=False)

train_df = manifest[manifest.split == "train"].reset_index(drop=True)
val_df = manifest[manifest.split == "val"].reset_index(drop=True)
test_df = manifest[manifest.split == "test"].reset_index(drop=True)

## 6. Pipeline and augmentation

Body-site photos come in any orientation, so geometric augmentation is stronger than in Model A: both flips, rotation, zoom and translation. Colour changes are kept small, with no hue shift, because redness and pigment carry diagnostic signal.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15, fill_mode="reflect"),
    tf.keras.layers.RandomZoom((-0.15, 0.15), fill_mode="reflect"),
    tf.keras.layers.RandomTranslation(0.08, 0.08, fill_mode="reflect"),
    tf.keras.layers.RandomContrast(0.15),
    tf.keras.layers.RandomBrightness(0.10, value_range=(0, 255)),
], name="augment")


def load_image(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.cast(tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method="bilinear"), tf.float32)  # 0..255


def make_ds(df, training):
    ys = tf.one_hot(df["y"].values, NUM_CLASSES)
    ds = tf.data.Dataset.from_tensor_slices((df["clean_path"].values, ys))
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: (load_image(p), y), num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda x, y: (tf.clip_by_value(augment(x, training=True), 0, 255), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df, False), make_ds(test_df, False)

x0 = load_image(train_df.loc[0, "clean_path"])[None]
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for i, ax in enumerate(axes):
    img = x0[0] if i == 0 else tf.clip_by_value(augment(x0, training=True)[0], 0, 255)
    ax.imshow(tf.cast(img, tf.uint8)); ax.axis("off")
plt.show()

## 7. Model

In [ ]:
class_weights = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_df["y"].values)
class_weights = {i: float(min(w, 10.0)) for i, w in enumerate(class_weights)}
print({CLASSES[i]: round(w, 2) for i, w in class_weights.items()})


def build_model():
    base = getattr(tf.keras.applications, BACKBONE)(include_top=False, weights=PRETRAINED_WEIGHTS,
                                                     input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.35)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="conditions")(x)
    return tf.keras.Model(inputs, outputs, name=f"skyntri_disease_{BACKBONE.lower()}"), base


model, base = build_model()
LOSS = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)
METRICS = [tf.keras.metrics.CategoricalAccuracy(name="acc"),
           tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top2"),
           tf.keras.metrics.AUC(name="auc", multi_label=True, num_labels=NUM_CLASSES)]
CKPT = WORK_DIR / "best_disease.keras"


def callbacks(patience):
    return [
        tf.keras.callbacks.ModelCheckpoint(str(CKPT), monitor="val_auc", mode="max", save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=patience, restore_best_weights=True),
        tf.keras.callbacks.CSVLogger(str(WORK_DIR / "training_log.csv"), append=True),
    ]

## 8. Training (run this yourself)

In [ ]:
model.compile(optimizer=tf.keras.optimizers.AdamW(HEAD_LR, weight_decay=1e-4), loss=LOSS, metrics=METRICS)
model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, class_weight=class_weights, callbacks=callbacks(3))

base.trainable = True
for layer in base.layers[:-UNFREEZE_LAST_N_LAYERS]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
steps = FINE_TUNE_EPOCHS * max(1, math.ceil(len(train_df) / BATCH_SIZE))
model.compile(optimizer=tf.keras.optimizers.AdamW(
    tf.keras.optimizers.schedules.CosineDecay(FINE_TUNE_LR, steps, alpha=0.05), weight_decay=1e-5),
    loss=LOSS, metrics=METRICS)
model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, class_weight=class_weights, callbacks=callbacks(6))

log = pd.read_csv(WORK_DIR / "training_log.csv")
log[["loss", "val_loss"]].plot(title="Loss"); plt.show()
log[["auc", "val_auc"]].plot(title="AUC"); plt.show()

## 9. Operating points (validation set only)

* **Referral threshold:** the largest `P(Suspicious lesion)` threshold that still reaches `TARGET_REFERRAL_SENSITIVITY` on validation. The flag is raised whenever the probability reaches it, even if another class has the top score.
* **Abstention threshold:** if the top probability is below `abstain_below`, the app says it cannot assess the image confidently.

In [ ]:
def predict_probs(m, ds):
    return np.concatenate([m.predict_on_batch(x) for x, _ in ds], axis=0)


best = tf.keras.models.load_model(CKPT, compile=False)
p_val, y_val = predict_probs(best, val_ds), val_df["y"].values

s = CLASS_INDEX["Suspicious lesion"]
is_s = (y_val == s).astype(int)
if is_s.sum() > 0:
    fpr, tpr, thr = roc_curve(is_s, p_val[:, s])
    ok = np.where(tpr >= TARGET_REFERRAL_SENSITIVITY)[0]
    referral_threshold = float(np.clip(thr[ok[0]], 0.01, 0.99)) if len(ok) else 0.05
else:
    referral_threshold = 0.05
print("Referral threshold:", round(referral_threshold, 4))

conf = p_val.max(1); correct = p_val.argmax(1) == y_val
abstain_below = 0.0
for t in np.linspace(0.2, 0.9, 71):
    kept = conf >= t
    if kept.mean() < 0.5:
        break
    if kept.any() and correct[kept].mean() >= TARGET_SELECTIVE_ACCURACY:
        abstain_below = float(t); break
print("Abstain below max-probability:", round(abstain_below, 3), "| coverage:", round(float((conf >= abstain_below).mean()), 3))

cov, acc = [], []
for t in np.linspace(0, 0.95, 40):
    k = conf >= t; cov.append(k.mean()); acc.append(correct[k].mean() if k.any() else np.nan)
plt.plot(cov, acc, marker="."); plt.xlabel("coverage"); plt.ylabel("accuracy on kept"); plt.title("Selective prediction (val)"); plt.show()

## 10. Test evaluation

In [ ]:
p_test, y_test = predict_probs(best, test_ds), test_df["y"].values
y_pred = p_test.argmax(1)

rows = []
for k, cls in enumerate(CLASSES):
    t, p = (y_test == k), (y_pred == k)
    tp, fn, fp, tn = (t & p).sum(), (t & ~p).sum(), (~t & p).sum(), (~t & ~p).sum()
    rows.append({"class": cls, "support": int(t.sum()),
                 "sensitivity": tp / max(tp + fn, 1), "specificity": tn / max(tn + fp, 1),
                 "PPV": tp / max(tp + fp, 1), "NPV": tn / max(tn + fn, 1),
                 "f1": f1_score(t, p, zero_division=0),
                 "roc_auc": roc_auc_score(t, p_test[:, k]) if 0 < t.sum() < len(t) else np.nan})
per_class = pd.DataFrame(rows).set_index("class")
display(per_class.round(3))

present = sorted(set(y_test))
summary = {
    "test_images": int(len(y_test)),
    "accuracy": float((y_pred == y_test).mean()),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "top2_accuracy": top_k_accuracy_score(y_test, p_test, k=2, labels=np.arange(NUM_CLASSES)),
    "macro_roc_auc_ovr": float(np.nanmean(per_class["roc_auc"])),
}
is_s = (y_test == s)
flag = p_test[:, s] >= referral_threshold
summary["referral_sensitivity"] = float((flag & is_s).sum() / max(is_s.sum(), 1))
summary["referral_specificity"] = float((~flag & ~is_s).sum() / max((~is_s).sum(), 1))
kept = p_test.max(1) >= abstain_below
summary["coverage_after_abstention"] = float(kept.mean())
summary["accuracy_on_kept"] = float((y_pred[kept] == y_test[kept]).mean()) if kept.any() else float("nan")
print(classification_report(y_test, y_pred, labels=np.arange(NUM_CLASSES), target_names=CLASSES, zero_division=0))
pd.Series(summary).round(4)

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=np.arange(NUM_CLASSES), normalize="true")
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt=".2f", xticklabels=CLASSES, yticklabels=CLASSES, cmap="Blues")
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Confusion matrix (row-normalised)")
plt.xticks(rotation=40, ha="right"); plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 6))
for k, cls in enumerate(CLASSES):
    t = (y_test == k).astype(int)
    if 0 < t.sum() < len(t):
        f, tp_, _ = roc_curve(t, p_test[:, k]); plt.plot(f, tp_, label=f"{cls} ({roc_auc_score(t, p_test[:, k]):.2f})")
plt.plot([0, 1], [0, 1], "k--"); plt.legend(fontsize=8); plt.title("One-vs-rest ROC"); plt.show()


def ece(conf, correct, bins=10):
    e, edges = 0.0, np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)


summary["ece_top1"] = ece(p_test.max(1), (y_pred == y_test).astype(float))
print("ECE (top-1 confidence):", round(summary["ece_top1"], 4))

In [ ]:
# Bootstrap 95% CIs (resampling by group, so correlated images of one case move together)
def bootstrap(fn, n=1000):
    rng = np.random.default_rng(SEED)
    gidx = test_df.groupby("group").indices
    keys = list(gidx)
    vals = []
    for _ in range(n):
        pick = np.concatenate([gidx[k] for k in rng.choice(keys, len(keys), replace=True)])
        vals.append(fn(pick))
    return [float(np.nanpercentile(vals, 2.5)), float(np.nanpercentile(vals, 97.5))]


ci = {
    "balanced_accuracy": bootstrap(lambda i: balanced_accuracy_score(y_test[i], y_pred[i])),
    "macro_f1": bootstrap(lambda i: f1_score(y_test[i], y_pred[i], average="macro", zero_division=0)),
    "referral_sensitivity": bootstrap(lambda i: (flag[i] & is_s[i]).sum() / max(is_s[i].sum(), 1)),
    "referral_specificity": bootstrap(lambda i: (~flag[i] & ~is_s[i]).sum() / max((~is_s[i]).sum(), 1)),
}
for k, v in ci.items():
    print(f"{k}: {summary[k]:.3f}  95% CI {v[0]:.3f}-{v[1]:.3f}")

In [ ]:
# Fairness: performance by Fitzpatrick group
sub = []
for g, idx in test_df.groupby("fst").indices.items():
    if len(idx) < 15:
        continue
    st = is_s[idx]
    sub.append({"fitzpatrick": g, "n": len(idx),
                "balanced_accuracy": balanced_accuracy_score(y_test[idx], y_pred[idx]),
                "referral_sensitivity": (flag[idx] & st).sum() / st.sum() if st.sum() else np.nan,
                "suspicious_n": int(st.sum())})
fairness = pd.DataFrame(sub)
display(fairness)

# By source
display(pd.DataFrame([{"source": s_, "n": len(i), "balanced_accuracy": balanced_accuracy_score(y_test[i], y_pred[i])}
                      for s_, i in test_df.groupby("source").indices.items()]))

In [ ]:
# False-negative review for the referral class: suspicious lesions that were NOT flagged.
# These must be reviewed by a clinician before release.
fn_idx = np.where(is_s & ~flag)[0]
print("Missed suspicious lesions:", len(fn_idx))
if len(fn_idx):
    fig, axes = plt.subplots(1, min(8, len(fn_idx)), figsize=(3 * min(8, len(fn_idx)), 3.5))
    for ax, i in zip(np.atleast_1d(axes), fn_idx[:8]):
        ax.imshow(Image.open(test_df.loc[i, "clean_path"])); ax.axis("off")
        ax.set_title(f"{test_df.loc[i, 'condition']}\nP={p_test[i, s]:.2f} FST {test_df.loc[i, 'fst']}", fontsize=8)
    plt.show()
test_df.loc[fn_idx, ["clean_path", "condition", "source", "fst"]].to_csv(WORK_DIR / "missed_suspicious_for_review.csv", index=False)

In [ ]:
# External test on DDI (never used for training or thresholds)
ddi_meta = DDI_ROOT / "ddi_metadata.csv"
external = None
if ddi_meta.exists():
    ddi = pd.read_csv(ddi_meta)
    index = {p.name: p for p in DDI_ROOT.rglob("*.png")}
    ddi["path"] = ddi["DDI_file"].map(lambda f: str(index.get(f, "")))
    ddi = ddi[ddi["path"] != ""].reset_index(drop=True)
    ds = tf.data.Dataset.from_tensor_slices(ddi["path"].values).map(load_image).batch(BATCH_SIZE)
    p_ddi = np.concatenate([best.predict_on_batch(x) for x in ds])
    mal = ddi["malignant"].astype(str).str.lower().isin(["true", "1"]).values
    flag_ddi = p_ddi[:, s] >= referral_threshold
    rows = []
    for tone, idx in ddi.groupby("skin_tone").indices.items():
        m = mal[idx]
        rows.append({"skin_tone": tone, "n": len(idx), "malignant": int(m.sum()),
                     "sensitivity": (flag_ddi[idx] & m).sum() / max(m.sum(), 1),
                     "specificity": (~flag_ddi[idx] & ~m).sum() / max((~m).sum(), 1)})
    external = pd.DataFrame(rows)
    display(external)
else:
    print("DDI not provided - external test skipped (strongly recommended before release).")

## 11. Export for the Skyntri backend

| File | Put it at | Backend setting |
|---|---|---|
| `skin_disease_efficientnet.keras` | `backend/models/` | `DISEASE_MODEL_PATH=models/skin_disease_efficientnet.keras` |
| `skin_disease_metadata.json` | `backend/models/` | `DISEASE_MODEL_METADATA_PATH=models/skin_disease_metadata.json` |

Then, **only after clinical review**, set `DISEASE_SCREENING_ENABLED=true`.

In [ ]:
MODEL_FILE = EXPORT_DIR / "skin_disease_efficientnet.keras"
META_FILE = EXPORT_DIR / "skin_disease_metadata.json"
best.save(MODEL_FILE)
sha256 = hashlib.sha256(MODEL_FILE.read_bytes()).hexdigest()

metadata = {
    "task": "disease_screening",
    "model_version": MODEL_VERSION,
    "backbone": BACKBONE,
    "image_size": IMG_SIZE,
    "preprocessing": "RGB, bilinear resize to image_size, float32 0-255 (no /255)",
    "output": "softmax",
    "classes": CLASSES,
    "other_class": OTHER,
    "referral_classes": REFERRAL_CLASSES,
    "referral_threshold": referral_threshold,
    "abstain_below": abstain_below,
    "sha256": sha256,
    "dataset_version": DATASET_VERSION,
    "sources": sorted(manifest["source"].unique().tolist()),
    "split_sizes": {k: int((manifest.split == k).sum()) for k in ("train", "val", "test")},
    "test_metrics": {k: round(float(v), 4) for k, v in summary.items()},
    "test_metrics_ci95": ci,
    "per_class": per_class.round(4).reset_index().to_dict(orient="records"),
    "fairness_by_fitzpatrick": fairness.round(4).to_dict(orient="records"),
    "external_ddi": None if external is None else external.round(4).to_dict(orient="records"),
    "clinical_review": "PENDING - do not enable in production before dermatologist sign-off",
}
META_FILE.write_text(json.dumps(metadata, indent=2, default=float))
print("Saved", MODEL_FILE, META_FILE)

## 12. Safe prediction (same logic as the backend)

The order matters. **Referral comes first**: a probability at or above the referral threshold always produces "see a doctor", whatever the top class is. Next comes abstention. A result is never phrased as "you are fine".

In [ ]:
loaded = tf.keras.models.load_model(MODEL_FILE, compile=False)
meta = json.loads(META_FILE.read_text())


def predict_disease(path):
    with Image.open(path) as im:
        x = np.asarray(im.convert("RGB").resize((meta["image_size"],) * 2, Image.Resampling.BILINEAR), np.float32)[None]
    probs = loaded.predict(x, verbose=0)[0]
    classes = meta["classes"]
    top = int(np.argmax(probs))
    referral = any(probs[classes.index(c)] >= meta["referral_threshold"] for c in meta["referral_classes"])
    if referral:
        status, message = "referral", "Some features may need a doctor's assessment. Please see a dermatologist soon."
    elif probs[top] < meta["abstain_below"]:
        status, message = "uncertain", "We cannot confidently assess this image."
    elif classes[top] == meta["other_class"]:
        status, message = "no_supported_pattern", "No supported pattern recognised. This is not a medical all-clear."
    else:
        status, message = "pattern", f"Features may be consistent with {classes[top]}. This is not a diagnosis."
    return {"status": status, "message": message, "top_class": classes[top],
            "probabilities": {c: round(float(p), 3) for c, p in zip(classes, probs)}}


xb, _ = next(iter(test_ds))
assert np.allclose(best.predict_on_batch(xb), loaded.predict_on_batch(xb), atol=1e-5)
predict_disease(test_df.loc[0, "clean_path"])